In [1]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv


#This function will load all the variable from the .env file and will 
#make them available in the os.environ dictionary(env variables)
load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro, API KEY variable exists!")
else:
    raise ValueError("GROQ_API_KEY not found")

from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser



llm_groq = ChatOpenAI(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1")

Bro, API KEY variable exists!


# **CHAIN WITH CUSTOM RUNNABLE**

In [47]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie summarizer"),
    ("human","Please summarize the movie in brief:{input}")
])



In [48]:
#TASK2(LLM)

llm_groq = ChatOpenAI(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1")

In [21]:
#TASK3-(Str parser)

str_parser = StrOutputParser()


In [22]:
#TASK-1 [PROMPT]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a Linkedin post generator"),
    ("human","Create a post for the following text for Linkedin:{text}")

])
#TASK-2 [LLM]

llm_groq = ChatOpenAI(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1")

#TASK-3 [str parser]

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_groq | str_parser


# **Parallel Chain 2**

In [63]:
from langchain_core.runnables import RunnableParallel,RunnableLambda

In [64]:
def insta_chain(text:dict):
    #TASK-1 [PROMPT]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","Youa re a insta post generator"),
        ("human","Create a post for the following text for insta:{text}")
    
    ])
    #TASK-2 [LLM]
    llm_groq = ChatOpenAI(
        model="openai/gpt-oss-20b",
        temperature=0,
        api_key=os.environ.get("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1")
    
    #TASK-3 [str parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_groq | str_parser

    result = chain_insta.invoke(text)
    return result
    
insta_chain_runnable = RunnableLambda(insta_chain)
    
    

# **Final Orchestration**

In [65]:
final_chain = (
    prompt_template | 
    llm_groq |
    str_parser |
    dictionary_maker_runnable |
    RunnableParallel(branches = {"linkedin" : chain_linkedin,"instagram": insta_chain_runnable})
)

In [66]:
final_chain.invoke("KGF")

{'branches': {'linkedin': '🚀 **KGF (Kolar Gold Fields) – A Masterclass in Ambition, Strategy & Resilience** 🚀  \n\nWhen I watched *KGF*, I wasn’t just entertained – I was reminded of the raw, relentless drive that fuels true leadership. The film follows **Rocky (Yash)**, a street‑wise orphan who rises from poverty to become the most feared gangster in Karnataka’s underground gold mines. His journey is a vivid illustration of:\n\n1. **Strategic Infiltration** – Rocky gains the trust of the powerful mine owner **Dhanraj (Vijay Raghavendra)**, then flips the power dynamic. In business, building credibility with stakeholders is often the first step to driving change.\n\n2. **Navigating Complex Politics** – From rival gangs to mine politics, Rocky’s story shows that success requires understanding and managing multiple, sometimes conflicting, interests.\n\n3. **Protecting Your Core Values** – Amid the chaos, Rocky safeguards his love interest **Raveena (Raveena Tandon in the Hindi version)**

# **Chain as a Runnable**

In [67]:
# TASK -1 [Beautify Function]

def beautify(final_response:dict)->dict:

    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']
    
    return {"linkedin":linkedin_response,"instagram":instagram_response}

beautify_runnable = RunnableLambda(beautify)

#TASK - 2 [Final Chain]

#final_chain

#Beautified Chain

beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")



{'linkedin': '🚀 **What “Pushpa” (2021) Teaches Us About Leadership, Risk & Resilience**  \n\nSet against the rugged Seshachalam hills, *Pushpa* follows Allu\u202fArjun’s character, a charismatic smuggler who rises to become the kingpin of the illegal red‑sandalwood trade. While the film is a high‑octane thriller, it also offers a surprisingly rich case study for anyone navigating complex, high‑stakes environments.\n\n### 1️⃣ Ambition Driven by Purpose  \nPushpa’s rise isn’t just about power—it’s fueled by a deep love for Srivalli and a desire to protect his community from corrupt officials.  \n**Takeaway:** Clear purpose can amplify motivation and guide decision‑making, even when the stakes are high.\n\n### 2️⃣ Navigating Uncertainty & Betrayal  \nAs the government intensifies its crackdown, Pushpa faces betrayals from within his own crew and relentless police pursuit.  \n**Takeaway:** In volatile markets, trust is fragile. Build robust systems and maintain open communication to mitiga